## Uppgift 16: Prediktera diamantpriser

#### Importer och ladda in datan

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

df = pd.read_csv("data/diamonds.csv")
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


#### Bearbeta datan

"Unnamed: 0" är bara ett gammalt radindex, den droppar vi. Kolumnerna cut, color och clarity är kategoriska (text) och behöver bli numeriska innan vi kan träna en modell, det gör vi med dummy-variable-encoding.

In [2]:
df = df.drop(columns="Unnamed: 0")
df = pd.get_dummies(df, columns=["cut", "color", "clarity"], drop_first=True, dtype=int)
df.head()

,carat,depth,table,price,x,y,z,cut_Good,cut_Ideal,cut_Premium,...,color_H,color_I,color_J,clarity_IF,clarity_SI1,clarity_SI2,clarity_VS1,clarity_VS2,clarity_VVS1,clarity_VVS2
0,0.23,61.5,55.0,326,3.95,3.98,2.43,0,1,0,...,0,0,0,0,0,1,0,0,0,0
1,0.21,59.8,61.0,326,3.89,3.84,2.31,0,0,1,...,0,0,0,0,1,0,0,0,0,0
2,0.23,56.9,65.0,327,4.05,4.07,2.31,1,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0.29,62.4,58.0,334,4.20,4.23,2.63,0,0,1,...,0,1,0,0,0,0,0,1,0,0
4,0.31,63.3,58.0,335,4.34,4.35,2.75,1,0,0,...,0,0,1,0,0,1,0,0,0,0


#### Dela upp i X och y, samt train/validation/test

In [3]:
X = df.drop(columns="price")
y = df["price"]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=40)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=36)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (30206, 23)
Validation: (12946, 23)
Test: (10788, 23)


#### Träna två modeller

In [4]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

tree_reg = DecisionTreeRegressor(random_state=42)
hyperparams = {"max_depth": [3, 5, 7, 10, None]}

tree_gs = GridSearchCV(tree_reg, hyperparams, scoring="neg_root_mean_squared_error", cv=5)
tree_gs.fit(X_train, y_train)

print(tree_gs.best_params_)

{'max_depth': 10}


#### Utvärdera på valideringsdatan

In [5]:
lin_reg_pred = lin_reg.predict(X_val)
tree_pred = tree_gs.predict(X_val)

rmse_lin_reg = root_mean_squared_error(y_val, lin_reg_pred)
rmse_tree = root_mean_squared_error(y_val, tree_pred)

print("RMSE Linear Regression:", rmse_lin_reg)
print("RMSE Decision Tree:", rmse_tree)

RMSE Linear Regression: 1099.912902869526
RMSE Decision Tree: 864.4861933042702


#### Slutgiltig utvärdering på testdata

Beslutsträdet vann på valideringsdatan, så vi tränar om det på train+val innan vi testar det på testdatan.

In [6]:
best_model = DecisionTreeRegressor(max_depth=10, random_state=42)
best_model.fit(X_train_full, y_train_full)

test_pred = best_model.predict(X_test)
rmse_test = root_mean_squared_error(y_test, test_pred)

print("RMSE på testdatan:", rmse_test)

RMSE på testdatan: 883.8036720932863


#### Träna om på hela datasetet och spara modellen

In [7]:
import joblib

best_model.fit(X, y)  # tränar om på hela datasetet innan modellen sparas/produktionssätts

joblib.dump(best_model, "diamant_modell.pkl")
print("Modell sparad!")

Modell sparad!
